# 01b - Automatic Player Collection

The first data collection notebook used a small manually selected player list.

That sample was useful for testing the pipeline, but it is too small and biased for meaningful clustering.

This notebook automatically collects a larger and more diverse set of Old School RuneScape player names using random partial username searches.

After collecting player names, the notebook downloads public OSRS Hiscores data for each player and saves the result as a structured dataset.

## 1. Import required packages

The following packages are used for:

- API requests,
- data handling,
- random search term generation,
- waiting between requests,
- file path management.

In [9]:
import requests
import pandas as pd
import time
import random
import string

from pathlib import Path

## 2. Project paths

The collected dataset will be saved into the `data/processed` folder.

The folder structure follows the project layout used throughout the repository.

In [10]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_DIR

WindowsPath('C:/Projects/osrs-player-segmentation/data/processed')

## 3. Why pseudo-random player collection?

Using only top leaderboard players would introduce strong sampling bias.

Top-ranked players usually have very high levels and similar progression patterns, which would make clustering less meaningful.

Instead, this notebook uses random partial username searches to collect a more diverse set of players.

This is not a perfectly random sample of all OSRS players, but it is more suitable for this project than a top leaderboard sample.

## 4. Generate random username search terms

The player collection process starts by generating random short character combinations.

These terms will be used as partial username searches.

In [11]:
def generate_random_search_terms(
    n_terms=300,
    min_length=2,
    max_length=4,
    random_state=42
):
    """
    Generate random partial username search terms.

    Examples:
    - ab
    - x7
    - kq2
    - 9rs

    These terms are later used to search for OSRS player names.
    """
    random.seed(random_state)

    terms = set()
    characters = string.ascii_lowercase + string.digits

    while len(terms) < n_terms:
        length = random.randint(min_length, max_length)
        term = "".join(random.choice(characters) for _ in range(length))
        terms.add(term)

    return list(terms)

In [12]:
search_terms = generate_random_search_terms(
    n_terms=20,
    min_length=2,
    max_length=4,
    random_state=42
)

search_terms[:10]

['m80o', 'k8pk', 'rak', 'osfo', '6b', '8f1c', 'yr9', 'hbrp', 'xmec', 'ig']

## 5. Search players by partial username

This function sends a request to the Wise Old Man player search endpoint.

For each random search term, it returns matching player names.

In [18]:
def search_wom_players(username_query, limit=20, max_retries=3):
    url = "https://api.wiseoldman.net/v2/players/search"

    params = {
        "username": username_query,
        "limit": limit
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=15)

            if response.status_code == 429:
                wait_time = 10 * (attempt + 1)
                print(f"Rate limited for '{username_query}'. Waiting {wait_time} seconds...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()
            data = response.json()

            player_names = []

            for item in data:
                username = item.get("displayName") or item.get("username")

                if username:
                    player_names.append(username)

            return player_names

        except requests.RequestException as error:
            print(f"Search failed for query '{username_query}': {error}")
            return []

    print(f"Search skipped after retries: {username_query}")
    return []

In [19]:
test_names = search_wom_players("rs", limit=10)

len(test_names), test_names[:10]

(10,
 ['RsSuomi',
  'Rs Jamal',
  'Rs But Afk',
  'rsvp',
  'Rs Wiki',
  'RS SLAG',
  'Rs Kajcsa',
  'rsyux',
  'RS ape',
  'rs basherr'])

## 6. Collect pseudo-random player names

This function repeatedly generates search terms, queries matching players, and stores unique usernames.

The process stops when the target number of unique players is reached or when all search terms have been used.

In [20]:
def collect_pseudo_random_player_names(
    target_count=500,
    n_search_terms=500,
    results_per_term=20,
    sleep_seconds=0.2,
    random_state=42
):
    """
    Collect a diverse set of OSRS player names using random partial username searches.

    Parameters:
    - target_count: desired number of unique player names
    - n_search_terms: number of generated random search terms
    - results_per_term: maximum number of players returned per search term
    - sleep_seconds: waiting time between API requests
    - random_state: seed for reproducibility

    Returns:
    - list of unique player names
    """
    search_terms = generate_random_search_terms(
        n_terms=n_search_terms,
        min_length=2,
        max_length=4,
        random_state=random_state
    )

    collected_names = []

    for index, term in enumerate(search_terms, start=1):
        print(f"[{index}/{len(search_terms)}] Searching term: {term}")

        names = search_wom_players(
            username_query=term,
            limit=results_per_term
        )

        collected_names.extend(names)
        collected_names = list(dict.fromkeys(collected_names))

        print(f"Collected unique players: {len(collected_names)}")

        if len(collected_names) >= target_count:
            break

        time.sleep(sleep_seconds)

    return collected_names[:target_count]

## 7. Run player name collection

In this step, we collect the player names that will be used for Hiscores data collection.

The target sample size can be adjusted depending on runtime.

In [21]:
player_names = collect_pseudo_random_player_names(
    target_count=1000,
    n_search_terms=2000,
    results_per_term=20,
    sleep_seconds=1.0,
    random_state=42
)

len(player_names), player_names[:20]

[1/2000] Searching term: obb
Collected unique players: 20
[2/2000] Searching term: 7kd6
Collected unique players: 20
[3/2000] Searching term: xpo
Collected unique players: 40
[4/2000] Searching term: i42
Collected unique players: 49
[5/2000] Searching term: zv9d
Collected unique players: 49
[6/2000] Searching term: q3d
Collected unique players: 49
[7/2000] Searching term: ws
Collected unique players: 69
[8/2000] Searching term: dm
Collected unique players: 89
[9/2000] Searching term: 7f7
Collected unique players: 90
[10/2000] Searching term: q2y4
Collected unique players: 90
[11/2000] Searching term: w1d
Collected unique players: 98
[12/2000] Searching term: et32
Collected unique players: 98
[13/2000] Searching term: 9p7k
Collected unique players: 98
[14/2000] Searching term: tbz
Collected unique players: 107
[15/2000] Searching term: z5jr
Collected unique players: 107
[16/2000] Searching term: xi9d
Collected unique players: 107
[17/2000] Searching term: n6s
Collected unique players: 1

[123/2000] Searching term: s601
Collected unique players: 780
[124/2000] Searching term: g1i
Collected unique players: 791
[125/2000] Searching term: lnl
Rate limited for 'lnl'. Waiting 10 seconds...
Rate limited for 'lnl'. Waiting 20 seconds...
Rate limited for 'lnl'. Waiting 30 seconds...
Search skipped after retries: lnl
Collected unique players: 791
[126/2000] Searching term: ww5
Collected unique players: 791
[127/2000] Searching term: i9l
Collected unique players: 792
[128/2000] Searching term: iv4
Collected unique players: 797
[129/2000] Searching term: xs
Collected unique players: 817
[130/2000] Searching term: w48
Collected unique players: 823
[131/2000] Searching term: nb
Collected unique players: 843
[132/2000] Searching term: vy
Collected unique players: 863
[133/2000] Searching term: 9yv
Collected unique players: 863
[134/2000] Searching term: xx9
Collected unique players: 874
[135/2000] Searching term: c2q
Collected unique players: 875
[136/2000] Searching term: nz
Collect

(1000,
 ['Obbyy',
  'Obby Cape',
  'Obby Apples',
  'Obby Kenobi',
  'obbE x',
  'Obby200',
  'Obby z',
  'obby',
  'obbiz',
  'Obby wang',
  'obbenheimer',
  'Obbib',
  'obbie bobbie',
  'Obbbo',
  'Obby Jonatan',
  'ObbMan',
  'Obby Skid',
  'obby chkn',
  'ObbyLobster',
  'obbset'])

## 8. Save collected player names

The collected usernames are saved separately.

This makes the data collection process more reproducible and allows reusing the same player list later.

In [22]:
player_names_path = PROCESSED_DATA_DIR / "osrs_collected_player_names.csv"

pd.DataFrame({"player": player_names}).to_csv(
    player_names_path,
    index=False,
    encoding="utf-8"
)

player_names_path

WindowsPath('C:/Projects/osrs-player-segmentation/data/processed/osrs_collected_player_names.csv')

## 9. Define OSRS Hiscores skill structure

The OSRS Hiscores endpoint returns skill data in a fixed order.

Each skill row contains:

- rank,
- level,
- experience.

The skill list must match the order returned by the Hiscores endpoint.

In [23]:
SKILLS = [
    "overall",
    "attack",
    "defence",
    "strength",
    "hitpoints",
    "ranged",
    "prayer",
    "magic",
    "cooking",
    "woodcutting",
    "fletching",
    "fishing",
    "firemaking",
    "crafting",
    "smithing",
    "mining",
    "herblore",
    "agility",
    "thieving",
    "slayer",
    "farming",
    "runecraft",
    "hunter",
    "construction",
    "sailing"
]

## Define OSRS boss Hiscores structure

After the skill section, the Hiscores response contains activity and boss killcount rows.

Boss rows usually contain:

- rank,
- killcount.

In [32]:
BOSSES = [
    "abyssal_sire",
    "alchemical_hydra",
    "barrows_chests",
    "bryophyta",
    "callisto",
    "cerberus",
    "chambers_of_xeric",
    "chambers_of_xeric_challenge_mode",
    "chaos_elemental",
    "chaos_fanatic",
    "commander_zilyana",
    "corporeal_beast",
    "crazy_archaeologist",
    "dagannoth_prime",
    "dagannoth_rex",
    "dagannoth_supreme",
    "deranged_archaeologist",
    "general_graardor",
    "giant_mole",
    "grotesque_guardians",
    "hespori",
    "kalphite_queen",
    "king_black_dragon",
    "kraken",
    "kreearra",
    "kril_tsutsaroth",
    "mimic",
    "nightmare",
    "obor",
    "sarachnis",
    "scorpia",
    "skotizo",
    "the_gauntlet",
    "the_corrupted_gauntlet",
    "theatre_of_blood",
    "thermonuclear_smoke_devil",
    "tzkal_zuk",
    "tztok_jad",
    "venenatis",
    "vetion",
    "vorkath",
    "wintertodt",
    "zalcano",
    "zulrah"
]

## 10. Fetch OSRS Hiscores data

After collecting player names, this section downloads public OSRS Hiscores data for each player.

In [54]:
from urllib.parse import quote

def fetch_wom_player_details(player_name, max_retries=2):
    """
    Fetch player details from Wise Old Man API.
    Handles spaces/special characters and basic retry logic.
    """
    encoded_name = quote(player_name, safe="")
    url = f"https://api.wiseoldman.net/v2/players/{encoded_name}"

    headers = {
        "User-Agent": "osrs-player-segmentation-student-project/1.0"
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, timeout=15)

            if response.status_code == 200:
                return response.json()

            if response.status_code in [403, 429]:
                wait_time = 5 * (attempt + 1)
                print(f"Rate/permission issue for {player_name} ({response.status_code}). Waiting {wait_time}s...")
                time.sleep(wait_time)
                continue

            print(f"Failed: {player_name} - HTTP {response.status_code}")
            return None

        except requests.RequestException as error:
            print(f"Request error for {player_name}: {error}")
            return None

    print(f"Skipped after retries: {player_name}")
    return None

## 11. Parse Hiscores data

The raw Hiscores response is text-based.

This function converts the skill section into a structured Python dictionary.

In [55]:
def parse_wom_player_details(player_name, data):
    """
    Parse Wise Old Man PlayerDetails JSON into a flat row.

    The latestSnapshot object contains skills, bosses and activities.
    """
    player_data = {
        "player": player_name
    }

    latest_snapshot = data.get("latestSnapshot")

    if not latest_snapshot:
        return player_data

    data_block = latest_snapshot.get("data", {})

    for category_name in ["skills", "bosses", "activities"]:
        category = data_block.get(category_name, {})

        for metric_name, metric_values in category.items():
            prefix = metric_name.lower()

            if category_name == "skills":
                player_data[f"{prefix}_rank"] = metric_values.get("rank")
                player_data[f"{prefix}_level"] = metric_values.get("level")
                player_data[f"{prefix}_xp"] = metric_values.get("experience")

            elif category_name == "bosses":
                player_data[f"{prefix}_rank"] = metric_values.get("rank")
                player_data[f"{prefix}_kc"] = metric_values.get("kills")

            elif category_name == "activities":
                player_data[f"{prefix}_rank"] = metric_values.get("rank")
                player_data[f"{prefix}_score"] = metric_values.get("score")

    return player_data

## 12. Download Hiscores data for collected players

This step iterates through the collected player list and downloads Hiscores data for each player.

Invalid or unavailable player profiles are skipped.

In [56]:
records = []
failed_players = []

for index, player_name in enumerate(player_names, start=1):
    print(f"[{index}/{len(player_names)}] Fetching WOM details: {player_name}")

    data = fetch_wom_player_details(player_name)

    if data is None:
        failed_players.append(player_name)
        continue

    parsed_data = parse_wom_player_details(player_name, data)
    records.append(parsed_data)

    time.sleep(0.5)

df_hiscores = pd.DataFrame(records)

df_hiscores.head()

[1/1000] Fetching WOM details: Obbyy
[2/1000] Fetching WOM details: Obby Cape
[3/1000] Fetching WOM details: Obby Apples
[4/1000] Fetching WOM details: Obby Kenobi
[5/1000] Fetching WOM details: obbE x
[6/1000] Fetching WOM details: Obby200
[7/1000] Fetching WOM details: Obby z
[8/1000] Fetching WOM details: obby
[9/1000] Fetching WOM details: obbiz
[10/1000] Fetching WOM details: Obby wang
[11/1000] Fetching WOM details: obbenheimer
[12/1000] Fetching WOM details: Obbib
[13/1000] Fetching WOM details: obbie bobbie
[14/1000] Fetching WOM details: Obbbo
[15/1000] Fetching WOM details: Obby Jonatan
[16/1000] Fetching WOM details: ObbMan
[17/1000] Fetching WOM details: Obby Skid
[18/1000] Fetching WOM details: obby chkn
Rate/permission issue for obby chkn (429). Waiting 5s...
Rate/permission issue for obby chkn (429). Waiting 10s...
Skipped after retries: obby chkn
[19/1000] Fetching WOM details: ObbyLobster
Rate/permission issue for ObbyLobster (429). Waiting 5s...
[20/1000] Fetching WOM

[139/1000] Fetching WOM details: NvMe
[140/1000] Fetching WOM details: nvrpuloutevr
[141/1000] Fetching WOM details: nvlI
[142/1000] Fetching WOM details: NVNV
[143/1000] Fetching WOM details: Nvsh
[144/1000] Fetching WOM details: NvMy Madness
[145/1000] Fetching WOM details: Nvious7
[146/1000] Fetching WOM details: Nvision
[147/1000] Fetching WOM details: nvrpurplight
[148/1000] Fetching WOM details: nvm ur tits
[149/1000] Fetching WOM details: 82bangbang
[150/1000] Fetching WOM details: 82busyair
[151/1000] Fetching WOM details: 82borkloki
[152/1000] Fetching WOM details: 82balduract
[153/1000] Fetching WOM details: 82bringthor
[154/1000] Fetching WOM details: lligmah
Rate/permission issue for lligmah (429). Waiting 5s...
Rate/permission issue for lligmah (429). Waiting 10s...
Skipped after retries: lligmah
[155/1000] Fetching WOM details: LLIGMA
Rate/permission issue for LLIGMA (429). Waiting 5s...
Rate/permission issue for LLIGMA (429). Waiting 10s...
Skipped after retries: LLIGMA


Skipped after retries: 07dive
[271/1000] Fetching WOM details: 07 Flare
Rate/permission issue for 07 Flare (429). Waiting 5s...
Rate/permission issue for 07 Flare (429). Waiting 10s...
Skipped after retries: 07 Flare
[272/1000] Fetching WOM details: 07 Nihilist
Rate/permission issue for 07 Nihilist (429). Waiting 5s...
[273/1000] Fetching WOM details: 07 Hollyman
[274/1000] Fetching WOM details: 07Mana
[275/1000] Fetching WOM details: 07Ashley
[276/1000] Fetching WOM details: 07wan
[277/1000] Fetching WOM details: 07Chris
[278/1000] Fetching WOM details: 07 0
[279/1000] Fetching WOM details: 07 Preacher
[280/1000] Fetching WOM details: 07 tim
[281/1000] Fetching WOM details: 07 PRISMA
[282/1000] Fetching WOM details: 07 t bone
[283/1000] Fetching WOM details: 07Geno
[284/1000] Fetching WOM details: 07TroyScape
[285/1000] Fetching WOM details: 07 maisy
[286/1000] Fetching WOM details: 07 ezra
[287/1000] Fetching WOM details: 07 Account
[288/1000] Fetching WOM details: 07 Camry
[289/1000

[407/1000] Fetching WOM details: w510
Rate/permission issue for w510 (429). Waiting 5s...
Rate/permission issue for w510 (429). Waiting 10s...
Skipped after retries: w510
[408/1000] Fetching WOM details: 99bottin
Rate/permission issue for 99bottin (429). Waiting 5s...
Rate/permission issue for 99bottin (429). Waiting 10s...
Skipped after retries: 99bottin
[409/1000] Fetching WOM details: 99Bank Stand
Rate/permission issue for 99Bank Stand (429). Waiting 5s...
Rate/permission issue for 99Bank Stand (429). Waiting 10s...
Skipped after retries: 99Bank Stand
[410/1000] Fetching WOM details: 99Brad
Rate/permission issue for 99Brad (429). Waiting 5s...
[411/1000] Fetching WOM details: 99bnkstnding
[412/1000] Fetching WOM details: 99botslayer
[413/1000] Fetching WOM details: 99broke
[414/1000] Fetching WOM details: 99bigworm99
[415/1000] Fetching WOM details: 99brews
[416/1000] Fetching WOM details: 99batatah
[417/1000] Fetching WOM details: 99BOXES
[418/1000] Fetching WOM details: 99b0wcap3


[535/1000] Fetching WOM details: cwum10
[536/1000] Fetching WOM details: cw90
[537/1000] Fetching WOM details: Cwiy
[538/1000] Fetching WOM details: Cw o
[539/1000] Fetching WOM details: Cwend
[540/1000] Fetching WOM details: cwan10
[541/1000] Fetching WOM details: Cwars Prod
[542/1000] Fetching WOM details: cwazi
[543/1000] Fetching WOM details: cwunchyroll
[544/1000] Fetching WOM details: Cwizzle69
[545/1000] Fetching WOM details: CWRL
Rate/permission issue for CWRL (429). Waiting 5s...
Rate/permission issue for CWRL (429). Waiting 10s...
Skipped after retries: CWRL
[546/1000] Fetching WOM details: cwisp damage
Rate/permission issue for cwisp damage (429). Waiting 5s...
Rate/permission issue for cwisp damage (429). Waiting 10s...
Skipped after retries: cwisp damage
[547/1000] Fetching WOM details: Cwaber
Rate/permission issue for Cwaber (429). Waiting 5s...
Rate/permission issue for Cwaber (429). Waiting 10s...
Skipped after retries: Cwaber
[548/1000] Fetching WOM details: cwob
Rate/

[666/1000] Fetching WOM details: Ksw1sh3r
[667/1000] Fetching WOM details: ksw
[668/1000] Fetching WOM details: Kswizz
[669/1000] Fetching WOM details: Kswann
[670/1000] Fetching WOM details: kswan
[671/1000] Fetching WOM details: kswizzy
[672/1000] Fetching WOM details: kswan14
[673/1000] Fetching WOM details: KSWVEC
[674/1000] Fetching WOM details: kswilly bets
[675/1000] Fetching WOM details: Kswivel
[676/1000] Fetching WOM details: 5x7
[677/1000] Fetching WOM details: b2p teemo
[678/1000] Fetching WOM details: b2p v2
[679/1000] Fetching WOM details: b2p
[680/1000] Fetching WOM details: b2p brigade
[681/1000] Fetching WOM details: b2pure
[682/1000] Fetching WOM details: B2P Grim9
Rate/permission issue for B2P Grim9 (429). Waiting 5s...
Rate/permission issue for B2P Grim9 (429). Waiting 10s...
Skipped after retries: B2P Grim9
[683/1000] Fetching WOM details: b2p host
Rate/permission issue for b2p host (429). Waiting 5s...
Rate/permission issue for b2p host (429). Waiting 10s...
Skipp

Skipped after retries: xSweetDreamz
[800/1000] Fetching WOM details: xstuart
Rate/permission issue for xstuart (429). Waiting 5s...
[801/1000] Fetching WOM details: xSleepgoodx
[802/1000] Fetching WOM details: xsantcastx
[803/1000] Fetching WOM details: xSouls
[804/1000] Fetching WOM details: xScrafty
[805/1000] Fetching WOM details: Xsk1l3rX
[806/1000] Fetching WOM details: xSxAxMx
[807/1000] Fetching WOM details: xSach
[808/1000] Fetching WOM details: xSteelbeardx
[809/1000] Fetching WOM details: xsomethingxd
[810/1000] Fetching WOM details: xSnik
[811/1000] Fetching WOM details: xSlayer
[812/1000] Fetching WOM details: xShady
[813/1000] Fetching WOM details: xSynna
[814/1000] Fetching WOM details: xSeori
[815/1000] Fetching WOM details: xsoar
[816/1000] Fetching WOM details: xsaby
[817/1000] Fetching WOM details: xsmall
[818/1000] Fetching WOM details: W489
[819/1000] Fetching WOM details: w488 imp
[820/1000] Fetching WOM details: w48smage
Rate/permission issue for w48smage (429). W

Rate/permission issue for 2xUniverse (429). Waiting 10s...
Skipped after retries: 2xUniverse
[937/1000] Fetching WOM details: 2x50
Rate/permission issue for 2x50 (429). Waiting 5s...
Rate/permission issue for 2x50 (429). Waiting 10s...
Skipped after retries: 2x50
[938/1000] Fetching WOM details: 2x92is99
Rate/permission issue for 2x92is99 (429). Waiting 5s...
[939/1000] Fetching WOM details: 2xshine
[940/1000] Fetching WOM details: 2xLightning
[941/1000] Fetching WOM details: 2x Tychy
[942/1000] Fetching WOM details: 2x210
[943/1000] Fetching WOM details: 2x8
[944/1000] Fetching WOM details: 2x Drops
[945/1000] Fetching WOM details: 2x Enh Dr
[946/1000] Fetching WOM details: 2xRice0Veggi
[947/1000] Fetching WOM details: PG4GP
[948/1000] Fetching WOM details: Fuyaning
[949/1000] Fetching WOM details: Fuyan
[950/1000] Fetching WOM details: J4S
[951/1000] Fetching WOM details: J4my
[952/1000] Fetching WOM details: J4COBI
[953/1000] Fetching WOM details: J4ymond
[954/1000] Fetching WOM det

,player,overall_rank,overall_level,overall_xp,attack_rank,attack_level,attack_xp,defence_rank,defence_level,defence_xp,...,pvp_arena_rank,pvp_arena_score,soul_wars_zeal_rank,soul_wars_zeal_score,guardians_of_the_rift_rank,guardians_of_the_rift_score,colosseum_glory_rank,colosseum_glory_score,collections_logged_rank,collections_logged_score
0,Obbyy,3606.0,2376.0,1.055380e+09,60190.0,99.0,25735539.0,15708.0,99.0,32203282.0,...,-1.0,2500.0,-1.0,0.0,761269.0,31.0,108255.0,23054.0,20558.0,840.0
1,Obby Cape,6936.0,2376.0,5.371348e+08,21439.0,99.0,18904741.0,8059.0,99.0,24819279.0,...,-1.0,2126.0,959.0,39003.0,14201.0,483.0,43031.0,637.0,2665.0,1091.0
2,Obby Apples,93294.0,2278.0,9.500952e+08,98436.0,99.0,21090532.0,136.0,99.0,200000000.0,...,-1.0,2500.0,-1.0,0.0,-1.0,0.0,-1.0,0.0,-1.0,157.0
3,Obby Kenobi,1008395.0,1750.0,2.861366e+08,-1.0,1.0,40.0,-1.0,1.0,0.0,...,-1.0,2500.0,194209.0,315.0,645805.0,45.0,-1.0,0.0,-1.0,96.0
4,obbE x,23800.0,2376.0,5.214336e+08,218871.0,99.0,15566135.0,47096.0,99.0,22895521.0,...,-1.0,2500.0,-1.0,0.0,367807.0,127.0,31651.0,40148.0,51659.0,657.0


In [57]:
[column for column in df_hiscores.columns if column.endswith("_kc")][:30]

['abyssal_sire_kc',
 'alchemical_hydra_kc',
 'amoxliatl_kc',
 'araxxor_kc',
 'artio_kc',
 'barrows_chests_kc',
 'brutus_kc',
 'bryophyta_kc',
 'callisto_kc',
 'calvarion_kc',
 'cerberus_kc',
 'chambers_of_xeric_kc',
 'chambers_of_xeric_challenge_mode_kc',
 'chaos_elemental_kc',
 'chaos_fanatic_kc',
 'commander_zilyana_kc',
 'corporeal_beast_kc',
 'crazy_archaeologist_kc',
 'dagannoth_prime_kc',
 'dagannoth_rex_kc',
 'dagannoth_supreme_kc',
 'deranged_archaeologist_kc',
 'doom_of_mokhaiotl_kc',
 'duke_sucellus_kc',
 'general_graardor_kc',
 'giant_mole_kc',
 'grotesque_guardians_kc',
 'hespori_kc',
 'kalphite_queen_kc',
 'king_black_dragon_kc']

In [60]:
df_hiscores[df_hiscores["player"].str.lower() == "RFU"][
    ["player", "cerberus_kc"]
]

,player,cerberus_kc


## 13. Inspect collected Hiscores dataset

Before saving the dataset, basic checks are performed.

In [61]:
df_hiscores.shape

(874, 244)

In [62]:
df_hiscores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 874 entries, 0 to 873
Columns: 244 entries, player to collections_logged_score
dtypes: float64(243), object(1)
memory usage: 1.6+ MB


In [64]:
df_hiscores.head()

,player,overall_rank,overall_level,overall_xp,attack_rank,attack_level,attack_xp,defence_rank,defence_level,defence_xp,...,pvp_arena_rank,pvp_arena_score,soul_wars_zeal_rank,soul_wars_zeal_score,guardians_of_the_rift_rank,guardians_of_the_rift_score,colosseum_glory_rank,colosseum_glory_score,collections_logged_rank,collections_logged_score
0,Obbyy,3606.0,2376.0,1.055380e+09,60190.0,99.0,25735539.0,15708.0,99.0,32203282.0,...,-1.0,2500.0,-1.0,0.0,761269.0,31.0,108255.0,23054.0,20558.0,840.0
1,Obby Cape,6936.0,2376.0,5.371348e+08,21439.0,99.0,18904741.0,8059.0,99.0,24819279.0,...,-1.0,2126.0,959.0,39003.0,14201.0,483.0,43031.0,637.0,2665.0,1091.0
2,Obby Apples,93294.0,2278.0,9.500952e+08,98436.0,99.0,21090532.0,136.0,99.0,200000000.0,...,-1.0,2500.0,-1.0,0.0,-1.0,0.0,-1.0,0.0,-1.0,157.0
3,Obby Kenobi,1008395.0,1750.0,2.861366e+08,-1.0,1.0,40.0,-1.0,1.0,0.0,...,-1.0,2500.0,194209.0,315.0,645805.0,45.0,-1.0,0.0,-1.0,96.0
4,obbE x,23800.0,2376.0,5.214336e+08,218871.0,99.0,15566135.0,47096.0,99.0,22895521.0,...,-1.0,2500.0,-1.0,0.0,367807.0,127.0,31651.0,40148.0,51659.0,657.0


In [65]:
len(failed_players), failed_players[:20]

(126,
 ['obby chkn',
  'xPottz',
  'xpotatoex',
  'i42O',
  'ws daplug',
  'WS ZaIc',
  'WSedanOwner',
  'dmoz',
  'dm 4 pawjob',
  'dmb',
  'n6s8tb2an7p',
  '0rthru5',
  '0regy',
  'NVREVRLUCKY',
  'nvrsang',
  'nvda go up',
  'lligmah',
  'LLIGMA',
  'lligetlucky',
  'ICDAS'])

## 14. Save the automatically collected Hiscores dataset

The final dataset is saved into the processed data folder.

This file will be used as input for the data cleaning notebook.

In [66]:
output_path = PROCESSED_DATA_DIR / "osrs_hiscores_auto_sample.csv"

df_hiscores.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

output_path

WindowsPath('C:/Projects/osrs-player-segmentation/data/processed/osrs_hiscores_auto_sample.csv')

## Summary

This notebook collected a larger and more diverse OSRS player sample.

The main steps were:

- generating random partial username search terms,
- collecting player names from public player search results,
- removing duplicate usernames,
- downloading public OSRS Hiscores data,
- parsing skill rank, level and experience values,
- saving the final dataset for cleaning and clustering.

The collected sample is not a perfectly random representation of all OSRS players, but it reduces the strong bias that would come from using only top leaderboard players.